# Fink/LSST — Select one Dia Object from visitId list

## 1. Imports & configuration

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import glob
import time
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # répertoire des parquets existants

# Colonnes à re-fetcher depuis l'API (légères — seulement ce qui manque)
# r:diaSourceId est la clé de jointure avec les parquets existants
COLS_VISIT = "r:diaObjectId,r:diaSourceId,r:visit,r:detector,r:midpointMjdTai,r:band,r:x,r:y"


print(f"Data directory : {os.path.abspath(DIR_DATA)}")
print(f"Colonnes fetched : {COLS_VISIT}")

## 6. Load `visit_index.csv` — index complet pour le Butler Rubin

In [ ]:
# =========================
# CONFIGURATION
# =========================
VISIT_PATH = f"{DIR_DATA}/visit_index.csv"  # adapte si besoin

# mapping bande -> position Y
BAND_MAP = {"u": 0, "g": 1, "r": 2, "i": 3, "z": 4, "y": 5}


# =========================
# LOAD DATA
# =========================
def load_data(path=VISIT_PATH):
    df = pd.read_csv(path)

    # Normalisation minimale des noms de colonnes
    cols = {c.lower(): c for c in df.columns}

    # tentative de mapping robuste
    def get_col(possible_names):
        for name in possible_names:
            if name in cols:
                return cols[name]
        return None

    dia_col = get_col(["diaobjectid", "objectid"])
    mjd_col = get_col(["r:mjd", "r:midpointmjdtai", "r:jd"])
    band_col = get_col(["r:band", "r:filter"])
    visit_col = get_col(["r:visit"])
    det_col = get_col(["r:detector", "r:ccd"])

    if dia_col is None or mjd_col is None:
        raise ValueError("Colonnes minimales manquantes (diaObjectId, MJD)")

    df = df.rename(columns={dia_col: "diaObjectId", mjd_col: "mjd"})

    if band_col:
        df = df.rename(columns={band_col: "band"})
    else:
        df["band"] = "unknown"

    if visit_col:
        df = df.rename(columns={visit_col: "visit"})
    else:
        df["visit"] = None

    if det_col:
        df = df.rename(columns={det_col: "detector"})
    else:
        df["detector"] = None

    return df


# =========================
# PLOT FUNCTION
# =========================
def plot_dia_objects_old(df):
    unique_objects = df["diaObjectId"].unique()

    plt.figure(figsize=(10, 6))

    for obj in unique_objects:
        sub = df[df["diaObjectId"] == obj]

        y = [BAND_MAP.get(b, -1) for b in sub["band"]]

        plt.scatter(sub["mjd"], y, label=str(obj), alpha=0.6)

    plt.yticks(list(BAND_MAP.values()), list(BAND_MAP.keys()))
    plt.xlabel("MJD")
    plt.ylabel("Band")
    plt.title("DIA Objects timeline")
    plt.legend(fontsize=6, ncol=2)
    plt.tight_layout()
    plt.show()


def plot_dia_objects(df):
    unique_objects = df["diaObjectId"].unique()
    n_obj = len(unique_objects)

    plt.figure(figsize=(20, 7))

    # Palette riche (scalable)
    cmap = plt.cm.get_cmap("tab20", n_obj)  # jusqu'à 20 couleurs distinctes
    # alternative plus dense :
    # cmap = plt.cm.get_cmap("nipy_spectral", n_obj)

    for i, obj in enumerate(unique_objects):
        sub = df[df["diaObjectId"] == obj]
        y = [BAND_MAP.get(b, -1) for b in sub["band"]]

        plt.scatter(sub["mjd"], y, color=cmap(i), label=str(obj), alpha=0.7, s=20)

    plt.yticks(list(BAND_MAP.values()), list(BAND_MAP.keys()))
    plt.xlabel("MJD")
    plt.ylabel("Band")
    plt.title(f"DIA Objects timeline ({n_obj} objects)")

    # --- LÉGENDE AMÉLIORÉE ---
    ncol = int(np.ceil(n_obj / 10))  # ~10 objets par colonne

    plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=ncol, fontsize=7, frameon=False)

    plt.tight_layout(rect=[0, 0.1, 1, 1])  # laisse de la place pour la légende
    plt.show()


# =========================
# INTERACTIVE SELECTION (simple)
# =========================
def plot_single_object_simple(df, dia_object_id):
    sub = df[df["diaObjectId"] == dia_object_id]

    if sub.empty:
        print(f"Objet {dia_object_id} non trouvé")
        return

    y = [BAND_MAP.get(b, -1) for b in sub["band"]]

    plt.figure(figsize=(8, 4))
    plt.scatter(sub["mjd"], y, c="black")

    plt.yticks(list(BAND_MAP.values()), list(BAND_MAP.keys()))
    plt.xlabel("MJD")
    plt.ylabel("Band")
    plt.title(f"DIA Object {dia_object_id}")
    plt.grid(True)
    plt.show()


def plot_single_object(df, dia_object_id):
    import matplotlib.pyplot as plt

    sub = df[df["diaObjectId"] == dia_object_id]

    if sub.empty:
        print(f"Objet {dia_object_id} non trouvé")
        return

    # Palette physique des filtres
    band_colors = {"u": "purple", "g": "green", "r": "red", "i": "orange", "z": "brown", "y": "black"}

    plt.figure(figsize=(10, 4))

    for band, group in sub.groupby("band"):
        y = [BAND_MAP.get(band, -1)] * len(group)

        plt.scatter(
            group["mjd"],
            y,
            color=band_colors.get(band, "gray"),
            label=band,
            s=40,
            edgecolor="black",
            alpha=0.9,
        )

    # Axes
    plt.yticks(list(BAND_MAP.values()), list(BAND_MAP.keys()))
    plt.xlabel("MJD")
    plt.ylabel("Filter")

    # Titre plus informatif
    plt.title(f"DIA Object {dia_object_id} — temporal sampling by band")

    # Grille légère
    plt.grid(True, linestyle="--", alpha=0.3)

    # Légende propre
    plt.legend(title="Band", loc="upper right", frameon=False)

    # Amélioration visuelle
    plt.tight_layout()
    plt.show()


# =========================
# EXTRACTION FUNCTION
# =========================
def get_visit_detector_list(df, dia_object_id):
    sub = df[df["diaObjectId"] == dia_object_id]

    if sub.empty:
        print(f"Objet {dia_object_id} non trouvé")
        return None

    result = sub[["visit", "detector"]].drop_duplicates()

    return result


# =========================
# SCRIPT USAGE
# =========================
# if __name__ == "__main__":
#    df = load_data()

#    print("Colonnes disponibles:", df.columns.tolist())

# Plot global
#    plot_dia_objects(df)

# Exemple d'utilisation
#    example_id = df["diaObjectId"].iloc[0]

#    print(f"\nExemple DIA Object: {example_id}")

#    plot_single_object(df, example_id)

#    visits = get_visit_detector_list(df, example_id)
#    print("\nVisits / Detectors associés:")
#    print(visits)

In [ ]:
df = load_data()

In [ ]:
plot_dia_objects(df)

In [ ]:
# Exemple d'utilisation
example_id = df["diaObjectId"].iloc[0]
print(f"\nExemple DIA Object: {example_id}")

In [ ]:
plot_single_object(df, example_id)

In [ ]:
visits = get_visit_detector_list(df, example_id)
print("\nVisits / Detectors associés:")
print(visits)

In [ ]:
for count, row in enumerate(df.itertuples()):
    obj = row.diaObjectId
    mjd = row.mjd

    print(f"\nDIA Object: {obj}")
    plot_single_object(df, obj)
    if count > 10:
        break

In [ ]:
assert False

## 7. Extraction des visitId pour le DRP Rubin

Exemples d'utilisation de l'index pour sélectionner des visites à traiter.

In [ ]:
# ── Exemple 1 : toutes les visites d'un groupe calibrateur ────────────────────
TARGET_GROUP = "gaia_star_stable_hq"  # changer selon le groupe souhaité

if not df_index.empty and TARGET_GROUP in df_index["group"].values:
    df_group = df_index[df_index["group"] == TARGET_GROUP]
    visits_group = sorted(df_group["r:visit"].dropna().unique())
    print(f'Groupe "{TARGET_GROUP}" — {len(visits_group)} visites uniques :')
    print(visits_group[:20], "..." if len(visits_group) > 20 else "")
else:
    print(f'Groupe "{TARGET_GROUP}" non trouvé. Groupes disponibles :')
    if not df_index.empty:
        print(df_index["group"].unique().tolist())

In [ ]:
# ── Exemple 2 : toutes les visites, tous groupes confondus ────────────────────
if not df_index.empty:
    all_visits = sorted(df_index["r:visit"].dropna().unique())
    print(f"Toutes visites (tous groupes) : {len(all_visits)} uniques")
    print("Exemples :", all_visits[:10])

    # Distribution par bande
    print("\nVisites par bande :")
    print(df_index.groupby("r:band")["r:visit"].nunique().rename("n_visits_uniques"))

    # Distribution par groupe
    print("\nVisites par groupe :")
    print(
        df_index.groupby("group")["r:visit"].nunique().sort_values(ascending=False).rename("n_visits_uniques")
    )

In [ ]:
# ── Exemple 3 : extraire la liste (visitId, detector) pour le Butler ──────────
# Format attendu par le DRP Rubin : liste de (visitId, detector)

if not df_index.empty:
    # Pour les étoiles Gaia stables uniquement, bande r
    mask = df_index["group"].str.startswith("gaia_star_stable") & (df_index["r:band"] == "r")
    df_drp = (
        df_index[mask][["r:visit", "r:detector"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["r:visit", "r:detector"])
        .reset_index(drop=True)
    )
    print(f"Paires (visitId, detector) pour étoiles Gaia stables en bande r :")
    print(f"  {len(df_drp)} paires uniques")
    print(df_drp.head(20).to_string())

    # Sauvegarder pour usage direct
    drp_path = os.path.join(DIR_DATA, "drp_visit_detector_gaia_r.csv")
    df_drp.to_csv(drp_path, index=False)
    print(f"\nSauvegardé : {drp_path}")

In [ ]:
# ── Exemple 4 : snippet Butler Rubin prêt à l'emploi (USDF) ──────────────────

butler_snippet = """
# ── Snippet Butler Rubin (USDF) ───────────────────────────────────────────────
# À adapter selon votre collection et pipeline

import pandas as pd
from lsst.daf.butler import Butler

# Charger l'index
df_index = pd.read_csv("data_FINK_BLOCK_LC_01/visit_index.csv")

# Sélectionner les visites d'intérêt (ex: étoiles Gaia stables, bande r)
mask = (
    df_index["group"].str.startswith("gaia_star_stable") &
    (df_index["r:band"] == "r")
)
visits_r = sorted(df_index[mask]["r:visit"].dropna().unique().astype(int))

# Initialiser le Butler
repo   = "/repo/embargo"          # adapter à votre collection USDF
coll   = "LSSTComCam/runs/DRP/..."
butler = Butler(repo, collections=[coll])

# Récupérer les calexps pour ces visites
for visit_id in visits_r[:5]:    # tester sur 5 d'abord
    try:
        calexp = butler.get("calexp", visit=int(visit_id), detector=0)
        print(f"visit {visit_id} OK  shape: {calexp.image.array.shape}")
    except Exception as e:
        print(f"visit {visit_id} ERREUR: {e}")
# ──────────────────────────────────────────────────────────────────────────────
"""
print(butler_snippet)

## 8. Vérification du contenu des parquets enrichis

In [ ]:
# Charger un fichier _v2 de contrôle et afficher ses colonnes
v2_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))
if v2_files:
    sample_path = v2_files[0]
    df_check = pd.read_parquet(sample_path)
    print(f"Vérification : {os.path.basename(sample_path)}")
    print(f"Shape : {df_check.shape}")
    print(f"\nColonnes (avec taux de remplissage) :")
    for col in df_check.columns:
        non_null = df_check[col].notna().sum()
        print(f"  {col:50s}  non-null: {non_null}/{len(df_check)}")
    print(f"\nExemple (5 lignes, colonnes clés) :")
    show_cols = [
        c
        for c in [
            "diaObjectId",
            "r:diaSourceId",
            "r:visit",
            "r:detector",
            "r:band",
            "r:midpointMjdTai",
            "r:psfFlux",
            "r:x",
            "r:y",
        ]
        if c in df_check.columns
    ]
    print(df_check[show_cols].dropna(subset=["r:visit"]).head(5).to_string())
else:
    print("Aucun fichier _v2 trouvé.")